# PhoBERT v4: active learning on real-world comments

Setup: **GPU T4 x2**, **Internet On**, secret `HF_TOKEN`.
**Input:** the same private dataset as v1-v3 (the three `vfyr_*_cleaned.csv` files) **plus** a private
dataset containing `active_labeled_v4.csv` (821 newly labeled YouTube/VOZ comments, columns text,label).

The notebook rebuilds the v3 data, adds the active-learning batch to **train only**
(`scripts/add_active_labels.py`), trains PhoBERT and pushes it to the **`v4` branch** (`main` is untouched).

In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji cleanlab
import glob, os, shutil, json
os.makedirs("data/raw", exist_ok=True)
csvs = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if "active_labeled" not in os.path.basename(p)]
for split in ["train", "val", "test"]:
    matches = [p for p in csvs if split in os.path.basename(p).lower()]
    assert len(matches) == 1, f"Expected exactly one CSV for '{split}', found: {matches}"
    shutil.copy(matches[0], f"data/raw/{split}.csv")
os.makedirs("reports", exist_ok=True)
!python scripts/find_label_issues.py | tail -2
!python scripts/make_clean_splits.py --prune > /dev/null
!python scripts/add_vihsd.py > /dev/null
!python scripts/apply_relabels.py --src data/clean_vihsd --dst data/clean_vihsd_v3
active = glob.glob("/kaggle/input/**/active_labeled_v4.csv", recursive=True)
assert len(active) == 1, f"Upload active_labeled_v4.csv as a Kaggle dataset, found: {active}"
os.makedirs("data/active", exist_ok=True)
shutil.copy(active[0], "data/active/active_labeled_v4.csv")
!python scripts/add_active_labels.py --active data/active/active_labeled_v4.csv --src data/clean_vihsd_v3 --dst data/clean_vihsd_v4


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
!python scripts/train_phobert.py --data-dir data/clean_vihsd_v4 --no-class-weights --epochs 6 --output-dir models/v4 --tag _v4 --no-push

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
api.create_branch(repo_id, branch="v4", exist_ok=True)
api.upload_folder(folder_path="models/v4/best", repo_id=repo_id, revision="v4", commit_message="v4: + active learning batch")
print(f"https://huggingface.co/{repo_id}/tree/v4")

Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps(json.load(open("reports/phobert_results_v4.json")), ensure_ascii=False))